# Completed SO4 example

Recorded notebook execution for **TCGA-FD-A5BS**, dated 2026-11-05.
Version: **SO4**; refined D1 segmentation with separately obtained SAM3 base weights;
vision model: **Qwen3.6-27B-Q6_K**. The run used 2 checksum-verified source slides
and the pinned candidate ROIs. Anatomical site: **bladder / urothelial tissue**.

Code cells, execution counts and outputs are retained. Machine-specific output paths are
normalized, and repeated progress lines are condensed for display.

[View HTML report](https://zhaorui-tan.github.io/PathXAI/SO4/report.html) · [Report text](../SO4/generated_long_report.txt) ·
[Run your own case](../../PathXAI_report.ipynb)


# PathXAI: a TCGA patient to an HTML report

Run this notebook on your own NVIDIA GPU computer/server after following [installation](docs/INSTALLATION.md).
Choose SO1/Core or SO4/Full; both use refined D1 segmentation internally. This is real inference, not a stored-report viewer.
The public sample is the two-slide patient input for TCGA-FD-A5BS, with a documented
anatomical site supplied before inference. The source slides and pinned candidate regions produce one patient report. The exact slide inventory, sizes and checksums are pinned in `configs/sample_tcga.json`.

Obtain your own access to SAM3, CONCH, and PathXAI D1 on Hugging Face. SO4 additionally needs KEEP access.
Supply a vision-capable Qwen API endpoint/model and credentials. Selected ROI images are sent to that API.
Never save credentials in notebook cells or outputs. References are used only in the optional evaluation cell after generation.


In [1]:
from pathlib import Path
import os, sys, json
from getpass import getpass

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pathxai_tutorial').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter from the PathXAI_tutorial repository directory.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from pathxai_tutorial import download_patient, sample_metadata, prepare_sample_candidates, make_config, check_api, run_report
from pathxai_tutorial.models import download_models
print('Tutorial:', ROOT.name)


Tutorial: tutorial


## 1. Choose a version and configure paths

Set the segmentation interpreter to the `pathxai-segmentation-py312` environment from the installation guide.
The current notebook kernel runs the report/CONCH pipeline. The D1 repository is `RayTan/PathXAI-Seg`;
the published immutable revision is filled in below. Obtain your own access to the separate upstream models before downloading.
The optional local model configuration is documented in [the inference guide](docs/INFERENCE.md).


In [2]:
VERSION = os.environ.get('PATHXAI_VERSION', 'SO1')  # SO1 or SO4
SEGMENTATION_PYTHON = os.environ.get('PATHXAI_SEG_PYTHON', '/path/to/pathxai-segmentation-py312/bin/python')
D1_REPO_ID = os.environ.get('PATHXAI_D1_REPO', 'RayTan/PathXAI-Seg')
D1_REVISION = os.environ.get('PATHXAI_D1_REVISION', 'abfe61cb8e6e916b7226fd9dba077ef95ee9ee4f')  # immutable 40-character HF commit
QWEN_API_BASE = os.environ.get('QWEN_API_BASE', '')
QWEN_MODEL = os.environ.get('QWEN_MODEL', '')
MAX_SECONDS = int(os.environ.get('PATHXAI_MAX_SECONDS', '3600'))
LOCAL_MODEL_CONFIG = os.environ.get('PATHXAI_LOCAL_MODEL_CONFIG', '')

# This optional JSON file contains paths and endpoint/model settings, never keys.
local = json.loads(Path(LOCAL_MODEL_CONFIG).read_text()) if LOCAL_MODEL_CONFIG else {}
if 'api_key' in local or 'hf_token' in local:
    raise ValueError('Remove credentials from the local configuration; use environment variables.')


## 2. Credentials and model downloads

The adapter download contains our LoRA, learned prompts and necessary D1 auxiliary parameters.
SAM3 base weights and CONCH come from their original repositories under your own permissions.
No automatic model substitution is performed when access is unavailable.


In [3]:
api_key = os.environ.get('QWEN_API_KEY', '')
if not api_key and os.environ.get('PATHXAI_API_NO_KEY', '') != '1':
    api_key = getpass('Qwen API key (leave blank only if your endpoint needs none): ')

if local:
    config = make_config(**local, api_key=api_key)
else:
    if not D1_REPO_ID or not D1_REVISION:
        raise ValueError('Set D1_REPO_ID and D1_REVISION from the model release before downloading.')
    hf_token = os.environ.get('HF_TOKEN') or getpass('HF read token for your approved model access: ')
    assets = download_models(d1_repo_id=D1_REPO_ID, d1_revision=D1_REVISION,
                             version=VERSION, token=hf_token)
    config = make_config(segmentation_python=SEGMENTATION_PYTHON,
                         qwen_api=QWEN_API_BASE, qwen_model=QWEN_MODEL,
                         api_key=api_key, hf_token=hf_token, **assets)
print('Selected report version:', VERSION)


Selected report version: SO4


## 3. Download and validate the public TCGA patient

The downloader checks every pinned GDC record, file size, MD5 and SHA-256. Interrupted transfers can be resumed.
If you already have the slide set, set `PATHXAI_SAMPLE_DIR` to its parent directory (nested GDC UUID folders
are supported). Every expected slide must pass verification; an incomplete patient is never run silently.
For your own data, replace `slides`, `case_id` and `known_site` with your patient's slide list, identifier
and anatomical sampling site, and set `candidate_rois = None` and `known_site_source = "user"`
to generate candidate ROIs from your own slides. Do not pass a tumor diagnosis as `known_site`.
The included sample reconstructs its pinned candidate ROIs directly from the downloaded slides.


In [4]:
sample = sample_metadata()
slides = download_patient(ROOT / 'data', existing_dir=os.environ.get('PATHXAI_SAMPLE_DIR') or None)
case_id = sample['case_id']
known_site = sample['known_site']
known_site_source = sample['known_site_source']['pipeline_marker']
candidate_rois = prepare_sample_candidates(slides, ROOT / 'data' / 'sample_rois')
print('Patient:', case_id)
print('Known anatomical site:', known_site)
print('Verified slides:', len(slides))
for slide in slides:
    print(' -', slide.name)


verified candidate 1/20: TCGA-FD-A5BS-01Z-00-DX1.A7155CEB-37C9-4AD2-B229-B76F18814585_foundation_x19040_y0_l0_w4096_h4096
verified candidate 2/20: TCGA-FD-A5BS-01A-02-TSB.21A9159B-7DD1-4F26-951E-C99C004C1956_foundation_x15984_y5344_l0_w4096_h4096
verified candidate 3/20: TCGA-FD-A5BS-01Z-00-DX1.A7155CEB-37C9-4AD2-B229-B76F18814585_foundation_x24160_y2736_l0_w4096_h4096
verified candidate 4/20: TCGA-FD-A5BS-01A-02-TSB.21A9159B-7DD1-4F26-951E-C99C004C1956_foundation_x33392_y12512_l0_w4096_h4096
verified candidate 5/20: TCGA-FD-A5BS-01A-02-TSB.21A9159B-7DD1-4F26-951E-C99C004C1956_foundation_x36464_y14560_l0_w4096_h4096
verified candidate 6/20: TCGA-FD-A5BS-01Z-00-DX1.A7155CEB-37C9-4AD2-B229-B76F18814585_foundation_x23136_y7856_l0_w4096_h4096
verified candidate 7/20: TCGA-FD-A5BS-01A-02-TSB.21A9159B-7DD1-4F26-951E-C99C004C1956_foundation_x3696_y8416_l0_w4096_h4096
verified candidate 8/20: TCGA-FD-A5BS-01Z-00-DX1.A7155CEB-37C9-4AD2-B229-B76F18814585_foundation_x20064_y6832_l0_w4096_h4096
ve

## 4. Verify the configured API

This makes one small synthetic-image request. A successful probe checks connectivity only; it does not
guarantee the full scientific workflow will succeed. No sample WSI is sent in this step.


In [5]:
probe = check_api(config)
print(probe['message'])
if not probe['ok']:
    raise RuntimeError('API probe failed. Correct endpoint, model and key before running inference.')


Synthetic image request and JSON response verified.


## 5. Run report generation

One case and one version run at a time. Keep the kernel/server alive; progress is also written to disk.
The time budget cancels the owned worker and preserves partial artifacts. Re-running uses a new output directory.
The sample supplies its documented anatomical site explicitly, as the historical patient run did.
Only slide images, candidate regions, patient identifier and anatomical site are passed to inference; cancer-project labels,
reference diagnosis, grade and evaluation results are not supplied to the report generator.


In [6]:
result = run_report(config, slide_paths=slides, case_id=case_id, version=VERSION,
                    known_site=known_site, known_site_source=known_site_source,
                    candidate_rois=candidate_rois, max_seconds=MAX_SECONDS,
                    output_dir=os.environ.get('PATHXAI_OUTPUT_DIR') or None)
print('Report:', result['report_html'])
print('Report text:', result['report_text'])


[preflight] Checking engine configuration
[asset_preflight] Verifying pinned model asset identities
[wsi_preparation] Resolving candidate ROIs
[engine] Starting original FULL pipeline (SO4_grade_aux_keep)
[engine_log]     [1/10]   ROI evidence seeding (20 candidates) ...
[engine_log]     [1/10] → 3 seed ROIs selected (TCGA-FD-A5BS-01Z-00-DX1.A7155CEB-37C9-4AD2-B229-B76F18814585_foundation_x19040_y0_l0_w4096_h4096, TCGA-FD-A5BS-01A-02-TSB.21A9159B-7DD1-4F26-951E-C99C004C1956_foundation_x15984_y5344_l0_w4096_h4096, TCGA-FD-A5BS-01Z-00-DX1.A7155CEB-37C9-4AD2-B229-B76F18814585_foundation_x24160_y2736_l0_w4096_h4096)
[engine_log]     [2/10]   Site: known bladder / urothelial tissue (no LLM)
[engine_log]     [2/10] → bladder / urothelial tissue  [source=known_site:tcga_site_only, conf=high]
[engine_log]     [3/10]   Segmentation (MedSAM, 3 ROIs) ...
[engine_log]     [3/10] → available=True, 3 masks
[engine_log]     [4/10]   Morphology / seg-reader evidence — ABLATED (skipped)
[engine_log]   

## 6. Open/export the report

Keep the HTML together with its `assets/` directory. Download/copy the whole portable folder to share the report.
The raw scientific outputs and exact configuration/provenance are retained beside it.


In [7]:
from IPython.display import FileLink, display
report_html = Path(result['report_html'])
try:
    display(FileLink(str(report_html.relative_to(ROOT))))
except ValueError:
    print('Report outside the Jupyter root; open it locally:', report_html)
print('Portable report folder:', report_html.parent)


Report outside the Jupyter root; open it locally: ../SO4/report.html
Portable report folder: ../SO4


## 7. Optional automatic report evaluation

Provide an independently curated reference report for the same case. This sample download does not contain
a verified reference report. Do not generate a reference from the model's output or infer it from a TCGA project label.
Use `generated_long_report.txt`, not the entire HTML viewer (which also contains evidence and differential diagnoses).
See [evaluation](docs/EVALUATION.md) for field applicability, metrics, structured references and batch input.
The two physician scoring tools are described separately in [scoring](docs/SCORING.md).


In [8]:
REFERENCE_REPORT = None  # Path('/your/reference_report.txt')
if REFERENCE_REPORT is None:
    print('Evaluation skipped: supply an independent reference report first.')
else:
    from evaluation.run_evaluation import evaluate_paths
    metrics = evaluate_paths(Path(result['report_text']), Path(REFERENCE_REPORT))
    evaluation_path = Path(result['run_dir']) / 'evaluation_v4.json'
    evaluation_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False) + '\n')
    print('Evaluation saved:', evaluation_path)


Evaluation skipped: supply an independent reference report first.


## Saved generated report

The following text was copied from this run's `generated_long_report.txt`.

```text
Specimen/Site:
Bladder / Urothelial tissue

Diagnosis:
high-grade urothelial carcinoma

Tumor/Lesion Description:
The submitted tissue sections demonstrate a malignant neoplasm characterized by diffuse sheets and solid nests of neoplastic cells. The tumor exhibits extensive geographic necrosis. The architecture is disorganized, consistent with a high-grade process.

Key Microscopic Findings:
Microscopic examination reveals a high-grade malignancy. The tumor cells are small to medium-sized with scant cytoplasm, hyperchromatic nuclei, and a high nuclear-to-cytoplasmic ratio. The growth pattern is predominantly solid and cohesive. Scant stroma is visible due to high cellularity and necrosis.

Invasion/Necrosis/Margins:
Invasion: Invasive features are likely present given the solid growth pattern and necrosis, though the specific invasion interface is not clearly visualized in these ROIs.
Necrosis: Present, extensive geographic necrosis.
Margins: Not assessable from the provided images.

Immunohistochemistry/Ancillary Studies:
Not assessable from H&E images. Recommended stains to confirm urothelial lineage and rule out mimics include GATA3, p63, CK7, and CK20.

Stage/Grade/Prognostic Features:
Malignancy: malignant based on the locked final diagnosis; grade: high grade.
Grade: High grade.
Stage: Not assessable.
Prognostic Features: The presence of extensive geographic necrosis and high-grade cytologic features are associated with aggressive behavior.
Subtype: urothelial carcinoma.
Invasion: invasive features likely present given the solid growth pattern and necrosis, though specific invasion interface is not clearly visualized in these ROIs.

Final Summary:
The morphologic features are suggestive of high-grade urothelial carcinoma. The diagnosis is supported by the solid growth pattern, marked cytologic atypia, and extensive necrosis. However, definitive subtyping cannot be confirmed without ancillary studies. Differential diagnoses such as small cell carcinoma of the bladder or metastatic carcinoma remain possible and should be ruled out with immunohistochemistry. Clinical correlation is recommended.
```
